# Predicción de Entregas (Olist Delivery Prediction)

Pipeline modularizado de Machine Learning para predecir el tiempo de entrega de órdenes en días a partir de datos transaccionales, geoespaciales y logísticos.

In [ ]:
# Librerías estándar y de terceros
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Módulos del proyecto (instalados en modo editable via pyproject.toml)
from src.common.data import load_delivery_raw_data
from src.delivery.features import build_delivery_features, clean_outliers
from src.delivery.viz import (
    plot_permutation_importance,
    plot_real_vs_predicted,
    plot_residuals,
)

In [ ]:
# Carga de datasets relacionales mediante el módulo común
raw_data = load_delivery_raw_data()
orders = raw_data["orders"]
items = raw_data["items"]
products = raw_data["products"]
customers = raw_data["customers"]
sellers = raw_data["sellers"]
geo = raw_data["geo"]

print("Datasets cargados exitosamente:")
for name, df_table in raw_data.items():
    print(f" - {name}: {df_table.shape}")

## Preparación de datos e Ingeniería de Features

In [ ]:
# Construcción del dataset consolidado con features espaciales y temporales
df = build_delivery_features(orders, items, products, customers, sellers, geo)
print(f"Dataset unificado: {df.shape}")

## Limpieza de outliers

In [ ]:
# Analisis y Limpieza de Outliers (Percentil 99)
print(f"Registros antes de limpiar outliers: {len(df)}")

df_limpio = clean_outliers(df, quantile=0.99)

print(f"Registros despues de limpiar: {len(df_limpio)}")

# Actualizar el dataframe principal
df = df_limpio

## Creación de Pipeline y Comparativa de Modelos de Regresión

In [ ]:
# Preparacion de datos para modelado temporal (sin data leakage)
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

features = [
    "distance_km",
    "estimated_days",
    "total_weight_g",
    "total_volume_cm3",
    "total_freight",
    "n_items",
    "order_month",
    "is_same_state",
]
X = df[features]
y = df["target_days"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

preprocessor = ColumnTransformer(transformers=[("num", "passthrough", features)])

# Modelos a comparar
modelos = {
    "Dummy Regressor (Mediana)": DummyRegressor(strategy="median"),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=10, n_jobs=-1, random_state=42
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=300, random_state=42),
}

results = []

print(" === COMPARATIVA DE MODELOS ===")
for name, regressor in modelos.items():
    regressor_log = TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
    )
    pipeline = Pipeline([("pre", preprocessor), ("reg", regressor_log)])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({"Modelo": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    print(f"{name:<28} -> MAE: {mae:5.2f} | RMSE: {rmse:5.2f} | R2: {r2:7.4f}")

results_df = pd.DataFrame(results).sort_values(by="MAE")
print(f"\nEl mejor modelo es: {results_df['Modelo'].values[0]}")

In [ ]:
# Re-ejecución del mejor modelo para visualización
best_regressor = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(max_iter=300, random_state=42),
    func=np.log1p,
    inverse_func=np.expm1,
)
best_model = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", best_regressor)])
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

plot_real_vs_predicted(y_test, y_pred)

**Interpretación del Gráfico (Reales vs. Predicciones):**
- Cada punto representa una orden del conjunto de test. La línea roja discontinua representa la predicción perfecta ($y = \hat{y}$).
- **Rango central (0 a 25 días):** El modelo ajusta con alta precisión la masa principal de órdenes, agrupándose densamente a lo largo de la diagonal.
- **Entregas largas (> 25 días):** Se observa una subestimación sistemática (puntos por debajo de la diagonal), donde el modelo predice entre 15 y 25 días para órdenes reales de 30 a 50 días, producto de la regresión a la media típica de ensambles en distribuciones asimétricas.

## Importancia de variables (permutation) y análisis de residuos

In [ ]:
# Permutation importance: cuánto empeora el RMSE (en días) al permutar cada variable
imp = plot_permutation_importance(best_model, X_test, y_test, features)

**Interpretación del Gráfico (Permutation Feature Importance):**
- Mide cuántos días empeora el error (RMSE) al permutar (barajar al azar) los valores de cada variable en el test set.
- **Variables dominantes:** `distance_km` (distancia geográfica vendedor-cliente) y `estimated_days` (promesa comercial previa) son los factores más determinantes en la capacidad predictiva.
- **Variables secundarias:** `total_volume_cm3`, `total_weight_g`, `is_same_state`, `n_items` y `total_freight` aportan correcciones físicas y logísticas (como la fricción de cruce de fronteras estatales) necesarias sobre el tiempo de transporte.
- **Variables de bajo impacto:** `order_month` presenta baja importancia por permutación aislada en test, pero es fundamental para anclar la estacionalidad temporal del entrenamiento.

In [ ]:
# Residuos vs predicción: el "embudo" confirma la heterocedasticidad
plot_residuals(y_test, y_pred)

**Interpretación del Gráfico (Análisis de Residuos vs. Predicción):**
- Cada punto es un residuo ($e_i = y_{real} - \hat{y}_{pred}$) respecto al valor predicho; la línea horizontal roja en $0$ indica error nulo.
- **Estabilización de varianza:** La transformación logarítmica $\log(1+y)$ mantiene la dispersión de errores contenida y equilibrada en la masa central del volumen operativo.
- **Residuos positivos en colas:** Los valores predichos entre 15 y 25 días muestran residuos positivos crecientes correspondientes a envíos con retrasos extremos no capturados por las features disponibles.

**Diagnóstico Global y Conclusiones Técnicas:**

1. **Comportamiento Operativo Óptimo (0–25 días):**
   - El modelo con `TransformedTargetRegressor` logra un MAE de **3.75 días** y un $R^2$ de **0.2921**, optimizando la mediana condicional de entrega para el 95% del volumen transaccional de Olist.

2. **Fenómeno de Regresión hacia la Media en Colas:**
   - En envíos que superan los 30 días, el ensamble de árboles tiende a predecir hacia la media condicional (18–25 días), evitando errores cuadráticos gigantescos a costa de subestimar las demoras excepcionales.

3. **Recomendaciones para Entornos Productivos:**
   - **SLA / Promesa al Cliente:** Implementar **Regresión Cuantílica (Quantile Loss al percentil 90)** o incorporar un buffer de seguridad sobre la predicción base para garantizar que el 90%+ de las entregas lleguen a tiempo.
   - **Ingeniería de Features Externa:** Incorporar variables del transportista logístico (*carrier*) y eventos climáticos regionales para complementar `is_same_state` y `distance_km`.